# Multilabel Random Oversampling

Three independent outputs are trained with BCE-based loss. Validation chooses the checkpoint; test is used only at the end.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

sys.path.append(str(Path.cwd()))
from multilabel_utils import (
    CLASS_NAMES,
    LABEL_COLUMNS,
    MultilabelDataset,
    calculate_multilabel_metrics,
    classifier_transform,
    create_resnet18,
    get_device,
    predict_multilabel,
    set_seed,
    train_classifier,
)

SEED = 42
THRESHOLD = 0.5
EPOCHS = 5
set_seed(SEED)
device = get_device()
PROJECT_ROOT = Path.cwd().parents[1]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "multilabel"
MODEL_DIR = PROJECT_ROOT / "models" / "multilabel"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
print("Device:", device)
print("Split sizes:", len(train_df), len(val_df), len(test_df))

Device: mps
Split sizes: 1503 302 300


In [2]:
from torch.utils.data import WeightedRandomSampler

transform = classifier_transform()
positive_frequency = train_df[LABEL_COLUMNS].mean()
inverse_frequency = 1.0 / positive_frequency
sample_weights = train_df[LABEL_COLUMNS].mul(inverse_frequency).max(axis=1)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights.to_numpy(), dtype=torch.double),
    num_samples=len(train_df),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(
    MultilabelDataset(train_df, PROJECT_ROOT, transform),
    batch_size=32,
    sampler=sampler,
)
val_loader = DataLoader(MultilabelDataset(val_df, PROJECT_ROOT, transform), batch_size=32)
test_loader = DataLoader(MultilabelDataset(test_df, PROJECT_ROOT, transform), batch_size=32)

In [3]:
model = create_resnet18(len(LABEL_COLUMNS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
checkpoint_path = MODEL_DIR / "oversampling_best.pth"

history = train_classifier(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    checkpoint_path,
    epochs=EPOCHS,
)

Epoch 1/5 | Train loss: 0.2406 | Validation loss: 0.1722


Epoch 2/5 | Train loss: 0.0903 | Validation loss: 0.1137


Epoch 3/5 | Train loss: 0.0561 | Validation loss: 0.1421


Epoch 4/5 | Train loss: 0.0431 | Validation loss: 0.1289


Epoch 5/5 | Train loss: 0.0330 | Validation loss: 0.1155


In [4]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
true_labels, probabilities, predictions = predict_multilabel(
    model, test_loader, device, threshold=THRESHOLD
)

print(classification_report(
    true_labels,
    predictions,
    target_names=CLASS_NAMES,
    zero_division=0,
))

metrics = calculate_multilabel_metrics(true_labels, predictions, "Oversampling")
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(PROCESSED_DIR / "oversampling_metrics.csv", index=False)
metrics_df.round(4)

               precision    recall  f1-score   support

Surface Crack       1.00      0.97      0.98       276
 Delamination       1.00      0.81      0.89        26
      Pinhole       0.91      0.96      0.93        73

    micro avg       0.98      0.95      0.97       375
    macro avg       0.97      0.91      0.94       375
 weighted avg       0.98      0.95      0.97       375
  samples avg       0.97      0.97      0.97       375



,Model,Exact Match Accuracy,Hamming Loss,Micro F1,Macro F1,Surface Crack F1,Delamination F1,Pinhole F1
0,Oversampling,0.9233,0.0278,0.9663,0.9362,0.9816,0.8936,0.9333


## Interpretation

Focus on macro F1 and the minority-label F1 scores. Exact-match accuracy requires the entire three-value vector to be correct.